# P7 — Serving and the weekly report (gate N4)

**Question:** does the serving build run from the tracked data, and does the weekly page it feeds
come out the same when built twice from cold?

This notebook builds the serving artifacts (nowcaster, its class prior, returns by regime, asset
returns) in two **scratch** copies under the gitignored notebook scratch directory, runs the
weekly page on each, and asserts the two pages are byte-identical. It then shows the page and the
facts Glenn should check: the month scored, the STALE banner, the allocation mode and the
tripwire's per-signal states.

Nothing here writes a tracked file or a registry row: the serving fit is not a registry trial
(`NO_REGISTRY`, ruling 2026-09-28), and every checkpoint the run writes lands in the scratch copy.

**Prerequisites — run these first if the data below is missing or old:**

- `python scripts/build_platform_data.py` (the data spine, `daily_raw` and, from 08.2-03 on,
  `fred_daily_raw` for the tripwire's credit signal; FRED key plus network)


In [1]:
%matplotlib inline
import contextlib
import io
import logging
import shutil

import numpy as np
from IPython.display import Markdown, display

logging.basicConfig(level=logging.WARNING)

from trading_crab_lib.platform import plotting as pplot
from trading_crab_lib.platform.config import load_platform_config
from trading_crab_lib.platform.honesty.registry import total_trial_count
from trading_crab_lib.platform.report import serving, weekly
from trading_crab_lib.platform.tripwire import monitor

cfg = load_platform_config()
# Never display cfg itself (T-06-01): it carries the live FRED key.
safe_cfg = pplot.redacted_config(cfg)
print("allocation mode (config):", safe_cfg["report"]["allocation_mode"])
print("tripwire stale threshold (business days):", safe_cfg["tripwire"]["stale_business_days"])
trials_before = total_trial_count()
print("registry trials before:", trials_before)


allocation mode (config): no_regime
tripwire stale threshold (business days): 5
registry trials before: 48


## Two cold scratch builds

Each build starts from an empty scratch directory: the tracked platform checkpoints
(`monthly_features`, `monthly_raw`, `regime_labels`, `daily_raw`, `fred_daily_raw` when present)
are copied in, serving is fitted there, and the weekly page is written inside the same scratch
directory. "Identical twice" means two cold copies; a second run on the same copy would
legitimately differ in the "Last week" column, because it finds the first run's book.


In [2]:
pages, managers = {}, {}
for name in ("p7_a", "p7_b"):
    scratch = pplot.NOTEBOOK_SCRATCH_DIR / name
    shutil.rmtree(scratch, ignore_errors=True)
    with contextlib.redirect_stdout(io.StringIO()):  # the builder prints absolute artifact paths
        managers[name] = serving.build_scratch_serving(cfg, scratch)
        pages[name], path = weekly.build_weekly_page(cfg, managers[name], output_dir=scratch / "out")
    print(f"{name}: {path.relative_to(pplot.NOTEBOOK_SCRATCH_DIR)} ({len(pages[name])} characters)")

assert pages["p7_a"] == pages["p7_b"], "the two cold builds produced different pages"
print("identical twice:", pages["p7_a"] == pages["p7_b"])
assert total_trial_count() == trials_before, "a registry row was appended"
print("registry trials after:", total_trial_count())


p7_a: p7_a/out/weekly_report.md (3040 characters)


p7_b: p7_b/out/weekly_report.md (3040 characters)
identical twice: True
registry trials after: 48


## The page

In [3]:
display(Markdown(pages["p7_a"]))

# Trading-Crab Platform Weekly Report

## Regime View (suspended)

Regime view: suspended — the served nowcaster is input-independent (fixed in the regime rebuild); the allocation does not use it.

Scored as of 2026-08-31, the latest month observed in all 55 of the nowcaster's model columns (the newest row). Nothing is imputed.

1 distinct posterior vector across 233 complete months (2007-04-30 → 2026-08-31): the served model scored on every month observed in all 55 model columns, compared exactly (no rounding, no threshold). The served posterior does not depend on the features: it is the same every week.

## Crash Tripwire (advisory — changes no weight)

- Realized-vol spike (SPY): STALE — 0.80x short/baseline EWMA vol (trips above 1.50x), as of 2026-09-18, more than 5 business days before the run date 2026-10-05; refresh with `python scripts/build_platform_data.py`.
- Credit-spread velocity (BAA-AAA): GREEN — +3 bps widening (trips at +25 bps), as of 2026-09-30
- Drawdown from peak (SPY): STALE — -1.84% (trips at -10.00%), as of 2026-09-18, more than 5 business days before the run date 2026-10-05; refresh with `python scripts/build_platform_data.py`.

Escalation: UNKNOWN (1 of 3 signals current — not green)

Run date 2026-10-05. A signal older than 5 business days is STALE and a missing input UNAVAILABLE; neither counts as green. The tripwire changes no weight on this page.

## Scoreboard (static — last budgeted run)

| Leg | TLW 1972–2020 | MDD 1972–2020 |
|---|---|---|
| Regime tilt (strategy) | 3.8789 | -26.96% |
| No-regime ablation | 4.1124 | -20.60% |
| SPY buy & hold | 4.9900 | -51.09% |
| 60/40 | 4.5098 | -29.49% |
| Faber 10-mo SMA | 4.9759 | -23.38% |

Own-span TLW (backtest_kpi_table.parquet): Regime tilt (strategy) 3.8789; No-regime ablation 4.1124; SPY buy & hold 5.7471 (from 1962-05); 60/40 5.1187 (from 1962-05); Faber 10-mo SMA 5.7114 (from 1962-02). The table above puts all five legs on the same months, so its numbers are comparable; the own-span ones are not.

Run 2026-10-04 (registry tag 08.3-monthend-tilt-vs-ablation) · window 1972-01-31 → 2020-12-31 · 588 monthly steps · 10 bps

Caveat (E-07): scoreboard returns are month-end to month-end, except oil before 1986, which uses monthly-average WTI (a bounded exception, E-08; D-02).

## Target vs. Current — Trades Implied

**Allocation mode:** no_regime

### Target allocation

| Class | Ticker | Target % | Last week % | Change |
|---|---|---|---|---|
| equities | SPY | 34.5% | n/a | n/a |
| long_duration | TLT | 34.3% | n/a | n/a |
| gold | IAU | 19.5% | n/a | n/a |
| oil | USO | 11.7% | n/a | n/a |
| cash | FZFXX | 0.0% | n/a | n/a |

Target is the executed book after the no-trade band; last week is the book this report executed before this run (n/a when there was none).

Targets below are the EXECUTED book after the 5.0% no-trade band (design §5.3 bounded turnover, 08-A7.md): an asset whose target moved by no more than the band from its last executed weight keeps that weight.

_Target allocation cash residual: 0.0%_


## What to check

In [4]:
page = pages["p7_a"]
scored = next(line for line in page.splitlines() if line.startswith("Scored as of"))
print("scored:", scored.split(",")[0].removeprefix("Scored as of "))

if "## STALE DATA" in page:
    banner = page.split("## STALE DATA")[1].split("\n## ")[0]
    print("STALE monthly series:")
    for line in banner.splitlines():
        if line.startswith("- "):
            print("  ", line[2:])
else:
    print("STALE monthly series: none")

mode = next(line for line in page.splitlines() if line.startswith("**Allocation mode:**"))
print("mode:", mode.removeprefix("**Allocation mode:** "))

tripwire = monitor.evaluate_tripwire(cfg, managers["p7_a"], run_date=weekly._run_date())
print(f"tripwire (run date {tripwire['run_date'].date()}):")
for name, sig in tripwire["signals"].items():
    as_of = sig["as_of"].date() if sig["as_of"] is not None else "-"
    print(f"   {name:<16} {sig['state']:<12} as of {as_of}")
# The page's own escalation line: "none" appears only when all three signals are current.
print("  ", next(line for line in page.splitlines() if line.startswith("Escalation:")))


scored: 2026-08-31
STALE monthly series: none
mode: no_regime
tripwire (run date 2026-10-05):
   vol_spike        stale        as of 2026-09-18
   credit_velocity  green        as of 2026-09-30
   spy_drawdown     stale        as of 2026-09-18
   Escalation: UNKNOWN (1 of 3 signals current — not green)


Read the page as Glenn would on a Friday:

- **Scored as of** is the latest month complete in every model column; nothing is imputed.
- **Allocation mode** is `no_regime` (DECISIONS A-13): the target is the measured no-regime
  ablation leg, and the regime view is suspended (D-03).
- **Crash Tripwire**: each signal names its own as-of date. A signal older than 5 business days
  reads STALE, a missing input UNAVAILABLE, and the escalation reads UNKNOWN unless all three
  signals are current. Rebuilding the data on the Mac refreshes them.
- **Scoreboard**: the last budgeted run (8.3, 2026-10-04), all five legs on 1972–2020, with the E-07
  caveat.


## N4 Sign-Off

Glenn fills this in after reading the notebook top to bottom (D-15 style: a plain cell, no helper,
no ledger).

**Date:** 2026-10-05

**Verdict:** approve

**Reasoning:** everything looks correct
